# 05 — Ontology-informed machine learning

This notebook compares two predictive representations for short-term motor progression:

1. **Conventional representation**: baseline raw variables used directly as model inputs.
2. **Ontology-informed representation**: baseline variables reorganised into semantic domain features and concept-level aggregates derived from the curated mapping.

**Prediction target**:
- `binary_progression_label`
- positive class: `faster`
- negative class: `slow` + `intermediate`

**Inputs**:
- `output/trajectory_analysis/subject_progression_table.csv`
- `output/trajectory_analysis/progression_visit_level_table.csv`
- `mapping/ppmi_pdon_pmdo_mapping.csv`

**Main outputs**:
- baseline modelling dataset
- conventional and ontology-informed feature matrices
- cross-validated metrics
- confusion matrices
- feature summaries saved to `output/ontology_informed_ml/`


In [ ]:
!pip -q install pandas numpy matplotlib scikit-learn

## 1) Project root (Drive recommended)

In [ ]:
from pathlib import Path
import os

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/ppmi-ontology-alignment')
else:
    PROJECT_DIR = Path('/content/ppmi-ontology-alignment')

MAP_DIR = PROJECT_DIR / 'mapping'
OUT_DIR = PROJECT_DIR / 'output'
TRAJ_DIR = OUT_DIR / 'trajectory_analysis'
ML_DIR = OUT_DIR / 'ontology_informed_ml'
FIG_DIR = ML_DIR / 'figures'
for p in [MAP_DIR, OUT_DIR, TRAJ_DIR, ML_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

MAP_PATH = MAP_DIR / 'ppmi_pdon_pmdo_mapping.csv'
PROG_PATH = TRAJ_DIR / 'subject_progression_table.csv'
VISIT_PATH = TRAJ_DIR / 'progression_visit_level_table.csv'

print('PROJECT_DIR:', PROJECT_DIR)
print('MAP_PATH   :', MAP_PATH, 'exists=', MAP_PATH.exists())
print('PROG_PATH  :', PROG_PATH, 'exists=', PROG_PATH.exists())
print('VISIT_PATH :', VISIT_PATH, 'exists=', VISIT_PATH.exists())
print('ML_DIR     :', ML_DIR)

## 2) Load inputs

In [ ]:
import pandas as pd
import numpy as np

if not MAP_PATH.exists():
    raise FileNotFoundError(f'Missing mapping file: {MAP_PATH}')
if not PROG_PATH.exists():
    raise FileNotFoundError(f'Missing progression file: {PROG_PATH}')
if not VISIT_PATH.exists():
    raise FileNotFoundError(f'Missing visit-level file: {VISIT_PATH}')

mapping = pd.read_csv(MAP_PATH, dtype=str).fillna('')
progression = pd.read_csv(PROG_PATH)
visit_level = pd.read_csv(VISIT_PATH)

print('Mapping rows:', len(mapping))
print('Progression rows:', len(progression))
print('Visit-level rows:', len(visit_level))
print('Unique subjects in progression:', progression['PATNO'].astype(str).nunique())
print('Unique subjects in visit-level:', visit_level['PATNO'].astype(str).nunique())
progression.head(3)

## 3) Define baseline modelling dataset

This notebook uses the first visit of the fixed two-visit pair (`V04`) as the predictor time point and a binary version of the progression label as the outcome.

In [ ]:
def normalize_event_id(x):
    return str(x).strip().upper()

visit_level['PATNO'] = visit_level['PATNO'].astype(str).str.strip()
visit_level['EVENT_ID_norm'] = (
    visit_level['EVENT_ID_norm'].astype(str).str.strip().str.upper()
    if 'EVENT_ID_norm' in visit_level.columns
    else visit_level['EVENT_ID'].apply(normalize_event_id)
)

progression['PATNO'] = progression['PATNO'].astype(str).str.strip()
progression['progression_group'] = progression['progression_group'].astype(str).str.strip()

baseline = visit_level[visit_level['EVENT_ID_norm'] == 'V04'].copy()

for col in ['progression_group', 'annualised_updrs3_change',
            'progression_group_x', 'progression_group_y',
            'annualised_updrs3_change_x', 'annualised_updrs3_change_y']:
    if col in baseline.columns:
        baseline = baseline.drop(columns=[col])

baseline = baseline.merge(
    progression[['PATNO', 'progression_group', 'annualised_updrs3_change']],
    on='PATNO',
    how='inner'
)

baseline['binary_progression_label'] = baseline['progression_group'].apply(lambda x: 1 if str(x).strip().lower() == 'faster' else 0)

print('Baseline modelling rows:', len(baseline))
print('Unique subjects:', baseline['PATNO'].nunique())
print('Original class distribution:')
print(baseline['progression_group'].value_counts())
print('\nBinary class distribution:')
print(baseline['binary_progression_label'].value_counts())
baseline.head(3)

## 4) Define raw variables and semantic domains

In [ ]:
SUBJECT_META = ['SEX', 'EDUCYRS', 'fampd_bin', 'APOE']
VISIT_META = ['age_at_visit']
MOTOR_VARS = ['updrs1_score', 'updrs2_score', 'updrs3_score', 'updrs4_score', 'hy']
COGNITION_VARS = ['moca', 'cogstate', 'MCI_testscores']
BIOMARKER_VARS = ['abeta', 'ptau', 'asyn']
IMAGING_VARS = [
    'mia_lowput_expected',
    'MIA_CAUDATE_L', 'MIA_CAUDATE_R', 'MIA_CAUDATE_BILAT',
    'MIA_PUTAMEN_L', 'MIA_PUTAMEN_R', 'MIA_PUTAMEN_BILAT',
    'MIA_STRIATUM_L', 'MIA_STRIATUM_R', 'MIA_STRIATUM_BILAT'
]

RAW_FEATURES = SUBJECT_META + VISIT_META + MOTOR_VARS + COGNITION_VARS + BIOMARKER_VARS + IMAGING_VARS
RAW_FEATURES = [c for c in RAW_FEATURES if c in baseline.columns]

print('Raw features:', RAW_FEATURES)
print('N raw features:', len(RAW_FEATURES))

## 5) Build conventional feature matrix

The conventional representation uses baseline raw variables directly.

In [ ]:
def to_float_safe(x):
    s = str(x).strip()
    if s == '' or s.lower() == 'nan':
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan

X_raw = baseline[['PATNO'] + RAW_FEATURES].copy()

for c in RAW_FEATURES:
    X_raw[c] = X_raw[c].apply(to_float_safe)

if 'APOE' in baseline.columns:
    apoe_raw = baseline['APOE'].astype(str).str.lower().fillna('')
    X_raw['APOE_e4_present'] = apoe_raw.apply(lambda s: 1.0 if 'e4' in s else 0.0)
    if 'APOE' in X_raw.columns:
        X_raw = X_raw.drop(columns=['APOE'])

raw_feature_cols = [c for c in X_raw.columns if c != 'PATNO']
y = baseline['binary_progression_label'].copy()

print('X_raw shape:', X_raw[raw_feature_cols].shape)
print('y shape    :', y.shape)
X_raw.head(3)

## 6) Build ontology-informed feature matrix

The ontology-informed representation reorganises baseline variables into semantic domain and concept-level aggregates.

This step does not change the classifier; it changes the feature representation.

In [ ]:
X_onto = pd.DataFrame({'PATNO': baseline['PATNO'].astype(str).values})

numeric_baseline = baseline.copy()
for c in RAW_FEATURES:
    if c in numeric_baseline.columns:
        numeric_baseline[c] = numeric_baseline[c].apply(to_float_safe)

if 'SEX' in baseline.columns:
    X_onto['sex_numeric'] = baseline['SEX'].apply(to_float_safe)
if 'EDUCYRS' in baseline.columns:
    X_onto['education_years'] = baseline['EDUCYRS'].apply(to_float_safe)
if 'fampd_bin' in baseline.columns:
    X_onto['family_history_pd'] = baseline['fampd_bin'].apply(to_float_safe)
if 'age_at_visit' in baseline.columns:
    X_onto['age_at_visit'] = baseline['age_at_visit'].apply(to_float_safe)
if 'APOE' in baseline.columns:
    apoe_raw = baseline['APOE'].astype(str).str.lower().fillna('')
    X_onto['APOE_e4_present'] = apoe_raw.apply(lambda s: 1.0 if 'e4' in s else 0.0)

motor_cols = [c for c in MOTOR_VARS if c in numeric_baseline.columns]
cognition_cols = [c for c in COGNITION_VARS if c in numeric_baseline.columns]
biomarker_cols = [c for c in BIOMARKER_VARS if c in numeric_baseline.columns]
imaging_cols = [c for c in IMAGING_VARS if c in numeric_baseline.columns]

if motor_cols:
    X_onto['motor_mean'] = numeric_baseline[motor_cols].mean(axis=1, skipna=True)
    X_onto['motor_sum'] = numeric_baseline[motor_cols].sum(axis=1, skipna=True)
if cognition_cols:
    X_onto['cognition_mean'] = numeric_baseline[cognition_cols].mean(axis=1, skipna=True)
if biomarker_cols:
    X_onto['biomarker_mean'] = numeric_baseline[biomarker_cols].mean(axis=1, skipna=True)
if imaging_cols:
    X_onto['imaging_mean'] = numeric_baseline[imaging_cols].mean(axis=1, skipna=True)

caudate_cols = [c for c in ['MIA_CAUDATE_L', 'MIA_CAUDATE_R', 'MIA_CAUDATE_BILAT'] if c in numeric_baseline.columns]
putamen_cols = [c for c in ['MIA_PUTAMEN_L', 'MIA_PUTAMEN_R', 'MIA_PUTAMEN_BILAT'] if c in numeric_baseline.columns]
striatum_cols = [c for c in ['MIA_STRIATUM_L', 'MIA_STRIATUM_R', 'MIA_STRIATUM_BILAT'] if c in numeric_baseline.columns]

if caudate_cols:
    X_onto['caudate_mean'] = numeric_baseline[caudate_cols].mean(axis=1, skipna=True)
if putamen_cols:
    X_onto['putamen_mean'] = numeric_baseline[putamen_cols].mean(axis=1, skipna=True)
if striatum_cols:
    X_onto['striatum_mean'] = numeric_baseline[striatum_cols].mean(axis=1, skipna=True)

X_onto['motor_n_obs'] = numeric_baseline[motor_cols].notna().sum(axis=1) if motor_cols else 0
X_onto['cognition_n_obs'] = numeric_baseline[cognition_cols].notna().sum(axis=1) if cognition_cols else 0
X_onto['biomarker_n_obs'] = numeric_baseline[biomarker_cols].notna().sum(axis=1) if biomarker_cols else 0
X_onto['imaging_n_obs'] = numeric_baseline[imaging_cols].notna().sum(axis=1) if imaging_cols else 0

onto_feature_cols = [c for c in X_onto.columns if c != 'PATNO']
print('Ontology-informed features:', onto_feature_cols)
print('N ontology-informed features:', len(onto_feature_cols))
X_onto.head(3)

## 7) Prepare models and cross-validation

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, make_scorer, precision_score, recall_score, f1_score

class_counts = y.value_counts()
min_class_size = int(class_counts.min())
n_splits = min(5, min_class_size)
if n_splits < 2:
    raise ValueError('Not enough samples per class for stratified cross-validation.')

cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

scoring = {
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced_accuracy',
    'f1': make_scorer(f1_score, zero_division=0),
    'precision': make_scorer(precision_score, zero_division=0),
    'recall': make_scorer(recall_score, zero_division=0)
}

models = {
    'svm_linear': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', LinearSVC(class_weight='balanced', random_state=42, max_iter=5000))
    ]),
    'rf': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42))
    ])
}

print('Cross-validation splits:', n_splits)
print('Binary class counts:')
print(class_counts)

## 8) Evaluate conventional and ontology-informed representations

In [ ]:
results = []
predictions = {}

feature_sets = {
    'conventional': (X_raw[raw_feature_cols], raw_feature_cols),
    'ontology_informed': (X_onto[onto_feature_cols], onto_feature_cols)
}

for rep_name, (Xmat, feature_names) in feature_sets.items():
    for model_name, model in models.items():
        cv_res = cross_validate(model, Xmat, y, cv=cv, scoring=scoring, return_train_score=False)
        y_pred = cross_val_predict(model, Xmat, y, cv=cv)
        predictions[(rep_name, model_name)] = y_pred
        results.append({
            'representation': rep_name,
            'model': model_name,
            'n_features': len(feature_names),
            'cv_accuracy_mean': float(np.mean(cv_res['test_accuracy'])),
            'cv_accuracy_std': float(np.std(cv_res['test_accuracy'])),
            'cv_balanced_accuracy_mean': float(np.mean(cv_res['test_balanced_accuracy'])),
            'cv_balanced_accuracy_std': float(np.std(cv_res['test_balanced_accuracy'])),
            'cv_f1_mean': float(np.mean(cv_res['test_f1'])),
            'cv_f1_std': float(np.std(cv_res['test_f1'])),
            'cv_precision_mean': float(np.mean(cv_res['test_precision'])),
            'cv_precision_std': float(np.std(cv_res['test_precision'])),
            'cv_recall_mean': float(np.mean(cv_res['test_recall'])),
            'cv_recall_std': float(np.std(cv_res['test_recall']))
        })

results_df = pd.DataFrame(results).sort_values(['model', 'representation']).reset_index(drop=True)
results_df

## 9) Confusion matrices

In [ ]:
import matplotlib.pyplot as plt

labels = [0, 1]
label_names = ['non_faster', 'faster']
cm_rows = []

for (rep_name, model_name), y_pred in predictions.items():
    cm = confusion_matrix(y, y_pred, labels=labels)
    cm_rows.append({
        'representation': rep_name,
        'model': model_name,
        'confusion_matrix': cm
    })

    plt.figure(figsize=(5, 4))
    plt.imshow(cm)
    plt.xticks(range(len(label_names)), label_names)
    plt.yticks(range(len(label_names)), label_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Confusion matrix: {rep_name} / {model_name}')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha='center', va='center')
    plt.tight_layout()
    out_path = FIG_DIR / f'confusion_matrix_{rep_name}_{model_name}.png'
    plt.savefig(out_path, dpi=200)
    plt.show()

print('Saved confusion matrices to:', FIG_DIR)

## 10) Simple feature inspection

This section reports feature names for both representations and fits the models once on the full dataset for exploratory inspection only. These fitted coefficients/importances are descriptive and should not be interpreted as out-of-sample validation results.

In [ ]:
feature_catalog = pd.DataFrame([
    {'representation': 'conventional', 'feature': f} for f in raw_feature_cols
] + [
    {'representation': 'ontology_informed', 'feature': f} for f in onto_feature_cols
])
feature_catalog.head(20)

In [ ]:
exploratory_outputs = []

for rep_name, (Xmat, feature_names) in feature_sets.items():
    svm_model = models['svm_linear']
    svm_model.fit(Xmat, y)
    clf_svm = svm_model.named_steps['clf']
    coef_abs = np.abs(clf_svm.coef_).reshape(-1)
    coef_df = pd.DataFrame({
        'representation': rep_name,
        'model': 'svm_linear',
        'feature': feature_names,
        'importance': coef_abs
    }).sort_values('importance', ascending=False)
    exploratory_outputs.append(coef_df)

    rf_model = models['rf']
    rf_model.fit(Xmat, y)
    clf_rf = rf_model.named_steps['clf']
    rf_df = pd.DataFrame({
        'representation': rep_name,
        'model': 'rf',
        'feature': feature_names,
        'importance': clf_rf.feature_importances_
    }).sort_values('importance', ascending=False)
    exploratory_outputs.append(rf_df)

feature_importance_df = pd.concat(exploratory_outputs, ignore_index=True)
feature_importance_df.head(20)

## 11) Export results

In [ ]:
baseline.to_csv(ML_DIR / 'baseline_modelling_dataset_binary.csv', index=False)
X_raw.to_csv(ML_DIR / 'conventional_feature_matrix_binary.csv', index=False)
X_onto.to_csv(ML_DIR / 'ontology_informed_feature_matrix_binary.csv', index=False)
results_df.to_csv(ML_DIR / 'cross_validated_metrics_binary.csv', index=False)
feature_catalog.to_csv(ML_DIR / 'feature_catalog_binary.csv', index=False)
feature_importance_df.to_csv(ML_DIR / 'exploratory_feature_importances_binary.csv', index=False)

cm_export_rows = []
for row in cm_rows:
    cm = row['confusion_matrix']
    for i, true_lab in enumerate(label_names):
        for j, pred_lab in enumerate(label_names):
            cm_export_rows.append({
                'representation': row['representation'],
                'model': row['model'],
                'true_label': true_lab,
                'predicted_label': pred_lab,
                'count': int(cm[i, j])
            })
confusion_export = pd.DataFrame(cm_export_rows)
confusion_export.to_csv(ML_DIR / 'confusion_matrices_binary.csv', index=False)

print('Wrote:')
for fn in [
    'baseline_modelling_dataset_binary.csv',
    'conventional_feature_matrix_binary.csv',
    'ontology_informed_feature_matrix_binary.csv',
    'cross_validated_metrics_binary.csv',
    'feature_catalog_binary.csv',
    'exploratory_feature_importances_binary.csv',
    'confusion_matrices_binary.csv'
]:
    print('-', ML_DIR / fn)

## 12) Quick interpretation-ready outputs

In [ ]:
print('--- cross_validated_metrics_binary ---')
print(results_df.round(3).to_string(index=False))

print('\n--- top exploratory feature importances (svm_linear/rf) ---')
for rep in ['conventional', 'ontology_informed']:
    for mdl in ['svm_linear', 'rf']:
        print(f'\n[{rep} / {mdl}]')
        sub = feature_importance_df[(feature_importance_df['representation'] == rep) & (feature_importance_df['model'] == mdl)].head(10)
        print(sub[['feature', 'importance']].round(4).to_string(index=False))